# 2. Tokenize transcriptomes and data pairing



This notebook starts with the CPU-only tokenization step. Replace placeholder paths like `path/to/...` with real locations on your system.


## 2.1. Configure paths and parameters

These parameters mirror the CLI options in `perturbgen.pp.GF_tokenisation`.


In order to download data, including the LPS data for this tutorial see https://perturbgen.cog.sanger.ac.uk/docs/data.html

In [1]:
import os
os.getcwd()

'/lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/docs/examples'

In [2]:
H5AD_PATH = "/lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/docs/examples/Public_LPS_PP.h5ad" #"path/to/data.h5ad"
DATASET_NAME = "LPS_all_tps_2k" # choose a name for the dataset
GENE_FILTERING_MODE = "hvg"  # one of: hvg, degs, all
HVG_MODE = "before_tokenisation"  # before_tokenisation or after_tokenisation

VAR_LIST = [
    "cell_type_harmonized",
    "time_after_LPS",
] # list of obs to retain in adata.vars after preprocessing

PAIRING_MODE = "stratified"  # stratified, random, mapping. We select the pairing strategy here, for more info please read the paper.
TIME_OBS = "time_after_LPS" # the obs that contains the time point information for pairing
PAIRING_FILE = None  # only if PAIRING_MODE == 'mapping'
MAIN_PAIRING_OBS = "cell_type_harmonized" # the main obs to use for pairing amongst TIME_OBS
OPT_PAIRING_OBS = []  # optional additional obs

NPROC = 8 # number of parallel processes to use
N_HVG = 2000 # number of highly variable genes to select if HVG filtering is used
TIME_POINT_ORDER = ["normal", "90m_LPS", "6h_LPS", "10h_LPS"]
REFERENCE_TIME = "normal" # the reference time point for pairing, usually the control or untreated condition

GENE_MEDIAN_PATH = "./T_perturb/perturbgen/pp/gene_median_dict_gftokens_gc95M.pkl"  # path to gene median dictionary, provided with Perturbgen for pretrained model
TOKEN_DICT_PATH  = "./T_perturb/perturbgen/pp/token_dict_gftokens_gc95M.pkl"  # path to token dictionary, provided with Perturbgen for pretrained model
GENE_MAPPING_PATH = "./T_perturb/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl"  # path to gene mapping dictionary, Geneformer 95M mapping dictionary provided with Perturbgen

In [3]:
%%bash  
ls ../../perturbgen/pp/

GF_tokenisation.py
__init__.py
__pycache__
ensembl_mapping_dict_gc95M.pkl
gene_median_dict_gftokens_gc95M.pkl
hspc
mpl_style.mplstyle
plt_tokenisation.py
token_dict_gftokens_gc95M.pkl
tokenizer.py


## 2.2. Build the tokenization command

This prints the exact command that will be executed. Review it before running.


In [4]:
cmd = [
    "python",
    "-m",
    "perturbgen",
    "tokenise",
    "--h5ad_path", H5AD_PATH,
    "--dataset", DATASET_NAME,
    "--gene_filtering_mode", GENE_FILTERING_MODE,
    "--hvg_mode", HVG_MODE,
    "--var_list", *VAR_LIST,
    "--pairing_mode", PAIRING_MODE,
    "--time_obs", TIME_OBS,
    "--main_pairing_obs", MAIN_PAIRING_OBS,
    "--nproc", str(NPROC),
    "--n_hvg", str(N_HVG),
    "--reference_time", REFERENCE_TIME,
    "--time_point_order", *TIME_POINT_ORDER,
    "--gene_median_path", GENE_MEDIAN_PATH,
    "--token_dict_path", TOKEN_DICT_PATH,
    "--gene_mapping_path", GENE_MAPPING_PATH,
]

if PAIRING_MODE == "mapping":
    cmd += ["--pairing_file", PAIRING_FILE]
if OPT_PAIRING_OBS:
    cmd += ["--opt_pairing_obs", *OPT_PAIRING_OBS]

print(" ".join(cmd))


python -m perturbgen tokenise --h5ad_path /lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/docs/examples/Public_LPS_PP.h5ad --dataset LPS_all_tps_2k --gene_filtering_mode hvg --hvg_mode before_tokenisation --var_list cell_type_harmonized time_after_LPS --pairing_mode stratified --time_obs time_after_LPS --main_pairing_obs cell_type_harmonized --nproc 8 --n_hvg 2000 --reference_time normal --time_point_order normal 90m_LPS 6h_LPS 10h_LPS --gene_median_path ./T_perturb/perturbgen/pp/gene_median_dict_gftokens_gc95M.pkl --token_dict_path ./T_perturb/perturbgen/pp/token_dict_gftokens_gc95M.pkl --gene_mapping_path ./T_perturb/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl


## 2.3. Run tokenization (CPU-only)

This step can take time depending on dataset size.


In [5]:
import subprocess

subprocess.run(cmd, check=True)

loading, please wait...
Current working directory: /lustre/scratch126/cellgen/lotfollahi/kl11
Start preprocessing adata...
Number of genes dropped: 0
Finished preprocessing adata.
Start tokenisation of adata...
Tokenizing /lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_GF_genes/LPS_all_tps_2k.h5ad


/lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/perturbgen/pp/tokenizer.py:495: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/perturbgen/pp/tokenizer.py:498: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_GF_genes/LPS_all_tps_2k.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.


Saving the dataset (1/1 shards): 100%|██████████| 104790/104790 [00:00<00:00, 1416475.82 examples/s]


Finished tokenisation.


...

Group ('CD14 monocytes',) has no cells, skipping...
Group ('CD16 monocytes',) has no cells, skipping...
Group ('Dendritic cells',) has no cells, skipping...
Group ('ILC1_3',) has no cells, skipping...
Group ('Plasmocytoid dendritic cell',) has no cells, skipping...
Group ('Red_Blood_Cell',) has no cells, skipping...
Group ('regulatory T cell',) has no cells, skipping...


  0%|          | 0/4 [00:00<?, ?it/s]/nfs/team361/cytomeister/.cytomeister/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)

 25%|██▌       | 1/4 [00:03<00:10,  3.55s/it]/nfs/team361/cytomeister/.cytomeister/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/nfs/team361/cytomeister/.cytomeister/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")

 50%|█████     | 2/4 [00:06<00:06,  3.39s/it]

No cell pairings found for time point 6h_LPS, skipping...


/nfs/team361/cytomeister/.cytomeister/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/nfs/team361/cytomeister/.cytomeister/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")

100%|██████████| 4/4 [00:10<00:00,  2.52s/it]


CompletedProcess(args=['python', '-m', 'perturbgen', 'tokenise', '--h5ad_path', '/lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/docs/examples/Public_LPS_PP.h5ad', '--dataset', 'LPS_all_tps_2k', '--gene_filtering_mode', 'hvg', '--hvg_mode', 'before_tokenisation', '--var_list', 'cell_type_harmonized', 'time_after_LPS', '--pairing_mode', 'stratified', '--time_obs', 'time_after_LPS', '--main_pairing_obs', 'cell_type_harmonized', '--nproc', '8', '--n_hvg', '2000', '--reference_time', 'normal', '--time_point_order', 'normal', '90m_LPS', '6h_LPS', '10h_LPS', '--gene_median_path', './T_perturb/perturbgen/pp/gene_median_dict_gftokens_gc95M.pkl', '--token_dict_path', './T_perturb/perturbgen/pp/token_dict_gftokens_gc95M.pkl', '--gene_mapping_path', './T_perturb/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl'], returncode=0)

## 2.4. Outputs

Tokenized files are written under the tokenized data directory defined in `perturbgen/configs/paths.py`.
You should see a new folder for your dataset name containing `.dataset` files and pairing outputs in root_dir/T_perturb/tokenized_data.


---
Next: training steps (masking model and count decoder) in the following sections. See Notebook 3 for training the Perturbgen model
